In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
net = nn.Sequential(nn.Flatten(),
                    nn.Linear(784, 256),
                    nn.ReLU(),
                    nn.Linear(256, 10))

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)

net.apply(init_weights);

In [3]:
batch_size, lr, num_epochs = 256, 0.1, 10
loss = nn.CrossEntropyLoss(reduction='none')
trainer = torch.optim.SGD(net.parameters(), lr=lr)

In [4]:
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)
d2l.train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)

AttributeError: module 'd2l.torch' has no attribute 'train_ch3'

In [7]:
import torch
from torch import nn
from d2l import torch as d2l

# 1. 数据加载（不变）
batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

# 2. 定义 MLP 模型类（继承 nn.Module，解决 train()/eval() 问题）
class MLP(nn.Module):
    def __init__(self, num_inputs, num_hiddens, num_outputs):
        super(MLP, self).__init__()  # 调用父类构造函数
        # 初始化参数（与原代码逻辑一致）
        self.W1 = nn.Parameter(torch.randn(num_inputs, num_hiddens, requires_grad=True) * 0.01)
        self.b1 = nn.Parameter(torch.zeros(num_hiddens, requires_grad=True))
        self.W2 = nn.Parameter(torch.randn(num_hiddens, num_outputs, requires_grad=True) * 0.01)
        self.b2 = nn.Parameter(torch.zeros(num_outputs, requires_grad=True))
    
    # 前向传播（复现原 net 函数逻辑）
    def forward(self, X):
        X = X.reshape((-1, num_inputs))  # 展平图像
        H = torch.max(X @ self.W1 + self.b1, torch.tensor(0.0))  # 内置 ReLU 逻辑（替代自定义 relu 函数）
        return H @ self.W2 + self.b2

# 3. 创建模型实例（参数与原代码一致）
num_inputs, num_outputs, num_hiddens = 784, 10, 256
net = MLP(num_inputs, num_hiddens, num_outputs)  # 现在 net 是 nn.Module 子类，支持 train()/eval()

# 4. 损失函数（不变）
loss = nn.CrossEntropyLoss(reduction='none')

# 5. 手动实现精度评估函数（不变）
def evaluate_accuracy(net, data_iter):
    net.eval()  # 现在可正常调用 eval() 切换评估模式
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in data_iter:
            y_hat = net(X)
            _, predicted = torch.max(y_hat, dim=1)
            total += y.size(0)
            correct += (predicted == y).sum().item()
    net.train()  # 切换回训练模式
    return correct / total

# 6. 训练循环（不变，现在 net.train() 可正常调用）
num_epochs, lr = 10, 0.1
updater = torch.optim.SGD(net.parameters(), lr=lr)  # 自动获取模型所有可训练参数

for epoch in range(num_epochs):
    net.train()  # 正常切换训练模式
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0
    
    for X, y in train_iter:
        y_hat = net(X)  # 调用模型的 forward 方法（无需手动调用）
        l = loss(y_hat, y).sum()
        
        updater.zero_grad()
        l.backward()
        updater.step()
        
        train_loss_sum += l.item()
        _, predicted = torch.max(y_hat, dim=1)
        train_total += y.size(0)
        train_correct += (predicted == y).sum().item()
    
    test_acc = evaluate_accuracy(net, test_iter)
    train_loss_avg = train_loss_sum / train_total
    train_acc = train_correct / train_total
    
    print(f"Epoch {epoch+1:2d} | "
          f"Train Loss: {train_loss_avg:.4f} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Test Acc: {test_acc:.4f}")

# 7. 预测可视化（尝试调用，报错则注释）
try:
    d2l.predict_ch3(net, test_iter)
except AttributeError:
    print("d2l.predict_ch3未找到，跳过可视化")

Epoch  1 | Train Loss: 447.6363 | Train Acc: 0.1012 | Test Acc: 0.1000
Epoch  2 | Train Loss: 3.6409 | Train Acc: 0.1006 | Test Acc: 0.1000
Epoch  3 | Train Loss: 2.9831 | Train Acc: 0.1003 | Test Acc: 0.1000
Epoch  4 | Train Loss: 3.0925 | Train Acc: 0.0991 | Test Acc: 0.1000
Epoch  5 | Train Loss: 3.0051 | Train Acc: 0.0998 | Test Acc: 0.1000
Epoch  6 | Train Loss: 3.0190 | Train Acc: 0.0988 | Test Acc: 0.1000
Epoch  7 | Train Loss: 2.9122 | Train Acc: 0.0999 | Test Acc: 0.1000
Epoch  8 | Train Loss: 2.9304 | Train Acc: 0.1002 | Test Acc: 0.1000
Epoch  9 | Train Loss: 2.9322 | Train Acc: 0.1008 | Test Acc: 0.1000
Epoch 10 | Train Loss: 2.9925 | Train Acc: 0.0987 | Test Acc: 0.1000
d2l.predict_ch3未找到，跳过可视化
